# Complete Design-State-1 Analysis: Exploring Design Condition Main Effects

This tutorial demonstrates how to analyze a fully replicated factorial design (design state 1) using the `by` parameter to explore different views of the same data.

## What You'll Learn

1. Load and formulate a two-factor study
2. Use the `by` parameter to aggregate Xbar/S charts at different levels
3. Stratify X charts by factor combinations
4. Understand lane boundaries in collapsed charts
5. Chart residuals using the `value` parameter

## Setup

In [1]:
from processbehavior import ProcessBehavior

## 1. Load and Formulate

We'll use the DS 1 validation dataset which has:
- **factor 1**: 3 levels (F1_1, F1_2, F1_3)
- **factor 2**: 2 levels (F2_1, F2_2)
- **time**: 8 periods
- Multiple replicates per cell

In [2]:
# Load the DS 1 validation data
pb = ProcessBehavior.read_csv('../../validation/sds1_data.csv')

print(f"Dataset: {pb.data.shape[0]} observations")
print(f"Factor 1 levels: {pb.data['factor 1'].unique().tolist()}")
print(f"Factor 2 levels: {pb.data['factor 2'].unique().tolist()}")
print(f"Time periods: {sorted(pb.data['time'].unique())}")
pb.data.head()

Dataset: 161 observations
Factor 1 levels: ['F1_1', 'F1_2', 'F1_3']
Factor 2 levels: ['F2_1', 'F2_2']
Time periods: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


,time,factor 1,factor 2,y
0,1,F1_1,F2_1,51.346824
1,1,F1_1,F2_1,51.516211
2,2,F1_1,F2_1,52.666367
3,2,F1_1,F2_1,52.699215
4,2,F1_1,F2_1,52.242386


In [3]:
# Formulate the study with both factors
study = pb.formulate(
    response='y',
    factors=['factor 1', 'factor 2'],
    time='time'
)

print(f"ADS: {study.analytical_design_state.sds} ({study.ads_reason})")
print(f"Valid charts: {study.valid_charts}")
print(f"Residual charts: {study.residual_charts}")

ADS: 1 (full_replication)
Valid charts: ['Histogram', 'Xbar', 'S', 'X', 'mR']
Residual charts: [('Xbar', 'R1'), ('X', 'R1'), ('S', 'R2'), ('X', 'R2'), ('Xbar', 'R3'), ('S', 'R3'), ('Xbar', 'R4'), ('S', 'R4'), ('Xbar', 'R5'), ('S', 'R5'), ('Xbar', 'R6'), ('S', 'R6')]


## 1.5 Quick Distribution Check: Histogram

Before diving into control charts, you can visualize the distribution of your response variable using a histogram.

In [4]:
# Histogram of response variable
study.execute(chart='Histogram', bins=15).plot(theme='ggplot').show()

In [5]:
study.execute(chart='Histogram', by=['factor 1'], bins=25).plot(theme='dark',show_zones=True).show()


You can also:
- Customize bins: `study.execute(chart='Histogram', bins=20)`
- Stratify by factors: `study.execute(chart='Histogram', by=['factor 1'])`
- Plot residual distributions: `study.execute(chart='Histogram', value='R5')`

## 2. Xbar Charts - Factor Aggregation

The `by` parameter controls how data points are aggregated on Xbar charts:
- **Default (all factors)**: One point per factor combination (6 points)
- **Single factor**: Aggregate across the other factor
- **Empty list**: Collapse to grand mean (1 point)

### 2.1 Xbar by All Factors (Default)

In [6]:
# Default: aggregate by all factors
result = study.execute(chart='Xbar')

print("Xbar chart data (one point per factor combination):")
result.get_chart('Xbar')

Xbar chart data (one point per factor combination):


,group,xbar,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,51.432,49.167,48.292,50.043,1
1,F1_1_F2_1_2,52.506,49.167,48.698,49.637,1
2,F1_1_F2_1_3,50.831,49.167,48.524,49.811,1
3,F1_1_F2_1_4,52.365,49.167,48.524,49.811,1
4,F1_1_F2_1_5,50.845,49.167,48.292,50.043,1
5,F1_1_F2_1_6,53.251,49.167,48.631,49.703,1
6,F1_1_F2_1_7,52.631,49.167,48.292,50.043,1
7,F1_1_F2_1_8,51.902,49.167,48.292,50.043,1
8,F1_1_F2_2_1,46.081,49.167,48.698,49.637,-1
9,F1_1_F2_2_2,47.795,49.167,48.292,50.043,-1


In [7]:
result.plot(chart='Xbar', show_stats=True).show()

### 2.2 Xbar by Factor 1 Only

In [8]:
# Aggregate by factor 1 only (3 points, one per F1 level)
result_f1 = study.execute(chart='Xbar', by=['factor 1'])

print("Xbar aggregated by factor 1:")
result_f1.get_chart('Xbar')

Xbar aggregated by factor 1:


,factor 1,xbar,center,lpl,upl,beyond_limits
0,F1_1,49.592,49.167,48.141,50.194,0
1,F1_2,47.380,49.167,48.188,50.147,-1
2,F1_3,50.543,49.167,48.141,50.194,1


In [9]:
result_f1.plot(chart='Xbar', show_stats=True).show()

### 2.3 Xbar by Factor 2 Only

In [10]:
# Aggregate by factor 2 only (2 points, one per F2 level)
result_f2 = study.execute(chart='Xbar', by=['factor 2'],)

print("Xbar aggregated by factor 2:")
result_f2.get_chart('Xbar')

Xbar aggregated by factor 2:


,factor 2,xbar,center,lpl,upl,beyond_limits
0,F2_1,51.257,49.167,48.579,49.755,1
1,F2_2,46.949,49.167,48.576,49.759,-1


In [11]:
result_f2.plot(chart='Xbar', show_stats=True).show()

### 2.4 Xbar Collapsed (Grand Mean)

In [12]:
# Collapse all factors (single point - grand mean)
result_all = study.execute(chart='Xbar', by=[])

print("Xbar collapsed to grand mean:")
result_all.get_chart('Xbar')
result_all.plot('Xbar').show()

Xbar collapsed to grand mean:


## 3. S Charts - Variation Analysis

S charts follow the same `by` parameter logic as Xbar charts.

In [13]:
# S chart by all factors (default)
result_s = study.execute(chart='S')

print("S chart (within-group standard deviation):")
result_s.get_chart('S')

S chart (within-group standard deviation):


,group,s,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,0.120,0.329,0.0,1.075,0
1,F1_1_F2_1_2,0.263,0.329,0.0,0.688,0
2,F1_1_F2_1_3,0.072,0.329,0.0,0.845,0
3,F1_1_F2_1_4,0.111,0.329,0.0,0.845,0
4,F1_1_F2_1_5,0.132,0.329,0.0,1.075,0
5,F1_1_F2_1_6,0.191,0.329,0.0,0.746,0
6,F1_1_F2_1_7,0.476,0.329,0.0,1.075,0
7,F1_1_F2_1_8,0.136,0.329,0.0,1.075,0
8,F1_1_F2_2_1,0.177,0.329,0.0,0.688,0
9,F1_1_F2_2_2,0.068,0.329,0.0,1.075,0


In [14]:
result_s.plot(chart='S', show_stats=True).show()

In [15]:
# S chart by factor 1 only
result_s_f1 = study.execute(chart='S', by=['factor 1'])

print("S chart aggregated by factor 1:")
result_s_f1.get_chart('S')

S chart aggregated by factor 1:


,factor 1,s,center,lpl,upl,beyond_limits
0,F1_1,2.457,2.455,1.724,3.186,0
1,F1_2,2.383,2.455,1.757,3.152,0
2,F1_3,2.524,2.455,1.724,3.186,0


In [16]:
# S chart by factor 2 only
result_s_f2 = study.execute(chart='S', by=['factor 2'])

print("S chart aggregated by factor 2:")
result_s_f2.get_chart('S')

S chart aggregated by factor 2:


,factor 2,s,center,lpl,upl,beyond_limits
0,F2_1,1.848,1.759,1.341,2.177,0
1,F2_2,1.669,1.759,1.338,2.179,0


## 4. X Charts - Stratified Analysis

X (individuals) charts with factors **require** an explicit `by` parameter. The `by` parameter controls stratification:
- **Both factors**: Separate chart for each factor combination
- **Single factor**: Charts per level with lane boundaries showing the other factor
- **Empty list**: Single chart with lane boundaries for all factor transitions

### 4.1 X by Both Factors (6 Faceted Charts)

In [17]:
# X stratified by both factors
result_imr = study.execute(chart='X', by=['factor 1', 'factor 2'])

print(f"Strata: {result_imr.charts['X']['strata']}")
print(f"Each stratum has its own X chart")

Strata: ['F1_1_F2_1', 'F1_1_F2_2', 'F1_2_F2_1', 'F1_2_F2_2', 'F1_3_F2_1', 'F1_3_F2_2']
Each stratum has its own X chart


In [18]:
result_imr.plot(chart='X', show_zones=True).show()

### 4.2 X by Factor 1 Only (3 Charts with Lane Boundaries)

When stratifying by one factor, the collapsed factor creates multiple observations at each time point. **Lane boundaries** show where the collapsed factor changes.

In [19]:
# X stratified by factor 1 only
result_imr_f1 = study.execute(chart='X', by=['factor 1'])

print(f"Strata: {result_imr_f1.charts['X']['strata']}")
print("\nLane boundaries show where factor 2 changes within each chart")

Strata: ['F1_1', 'F1_2', 'F1_3']

Lane boundaries show where factor 2 changes within each chart


In [20]:
result_imr_f1.plot(chart='X', show_zones=True).show()

### 4.3 X by Factor 2 Only (2 Charts with Lane Boundaries)

In [21]:
# X stratified by factor 2 only
result_imr_f2 = study.execute(chart='X', by=['factor 2'])

print(f"Strata: {result_imr_f2.charts['X']['strata']}")
print("\nLane boundaries show where factor 1 changes within each chart")

Strata: ['F2_1', 'F2_2']

Lane boundaries show where factor 1 changes within each chart


In [22]:
result_imr_f2.plot(chart='X', show_zones=True).show()

### 4.4 Single X Chart (Collapsed, with Lane Boundaries)

In [23]:
# Single X chart with all factors collapsed
result_imr_all = study.execute(chart='X', by=[])

print("Single X chart with all data")
print("Lane boundaries show transitions between factor combinations")

Single X chart with all data
Lane boundaries show transitions between factor combinations


In [24]:
result_imr_all.plot(chart='X', show_zones=True).show()

## 5. Residual Charts

Use the `value` parameter to chart VAS residuals instead of the response variable.

### 5.1 R5 (Design Condition Main Effects) on Xbar

In [25]:
# R5 residuals show factor effects
result_r5 = study.execute(chart='Xbar', value='R5')

print("R5 Xbar chart (factor effects):")
result_r5.get_chart('Xbar')

R5 Xbar chart (factor effects):


,group,xbar,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,2.803,-0.0,-0.875,0.875,1
1,F1_1_F2_1_2,2.803,-0.0,-0.470,0.470,1
2,F1_1_F2_1_3,2.803,-0.0,-0.643,0.643,1
3,F1_1_F2_1_4,2.803,-0.0,-0.643,0.643,1
4,F1_1_F2_1_5,2.803,-0.0,-0.875,0.875,1
5,F1_1_F2_1_6,2.803,-0.0,-0.536,0.536,1
6,F1_1_F2_1_7,2.803,-0.0,-0.875,0.875,1
7,F1_1_F2_1_8,2.803,-0.0,-0.875,0.875,1
8,F1_1_F2_2_1,-1.624,-0.0,-0.470,0.470,-1
9,F1_1_F2_2_2,-1.624,-0.0,-0.875,0.875,-1


In [26]:
result_r5.plot(chart='Xbar', show_stats=True).show()

### 5.2 Recentered Residuals

Use `recentered=True` to center residuals around zero.

In [27]:
# Recentered R5 residuals
result_r5_rc = study.execute(chart='Xbar', value='R5', recentered=True)

print("Recentered R5 Xbar chart:")
result_r5_rc.get_chart('Xbar')

Recentered R5 Xbar chart:


,group,xbar,center,lpl,upl,beyond_limits
0,F1_1_F2_1_1,51.432,49.17,48.294,50.045,1
1,F1_1_F2_1_2,52.506,49.17,48.700,49.639,1
2,F1_1_F2_1_3,50.831,49.17,48.526,49.813,1
3,F1_1_F2_1_4,52.365,49.17,48.526,49.813,1
4,F1_1_F2_1_5,50.845,49.17,48.294,50.045,1
5,F1_1_F2_1_6,53.251,49.17,48.634,49.705,1
6,F1_1_F2_1_7,52.631,49.17,48.294,50.045,1
7,F1_1_F2_1_8,51.902,49.17,48.294,50.045,1
8,F1_1_F2_2_1,46.081,49.17,48.700,49.639,-1
9,F1_1_F2_2_2,47.795,49.17,48.294,50.045,-1


In [28]:
result_r5_rc.plot(chart='Xbar', show_stats=True).show()

### 5.3 R5 on S Chart

In [29]:
# R5 on S chart

study.execute(chart='S', value='R5', recentered=True).plot().show()

## 6. Study Inspection

Before executing charts, you can inspect what the study supports. The `study.support` property shows all chart types, their availability, and the analytical question each answers.

In [30]:
# Full chart support matrix
study.support

,chart,value,category,available,recommended,reason,question
0,Xbar,None,primary,True,True,None,Are subgroup means stable over time?
1,S,None,primary,True,False,None,Is within-subgroup variation stable?
2,X,None,primary,True,False,None,Is individual variation stable over time?
3,mR,None,primary,True,False,None,Is range variation stable over time?
4,Histogram,None,primary,True,False,None,What shape does the response distribution take?
5,Xbar,R1,residual,True,False,None,How do subgroup means vary about the overall m...
6,X,R1,residual,True,False,None,How do individual values vary about the overal...
7,S,R2,residual,True,False,None,Is within-subgroup variation stable?
8,X,R2,residual,True,False,None,Is within-subgroup variation stable?
9,Xbar,R3,residual,True,False,None,Is there factor×time interaction affecting the...


In [31]:
# Filter to available charts only
study.support[study.support['available']]

,chart,value,category,available,recommended,reason,question
0,Xbar,None,primary,True,True,None,Are subgroup means stable over time?
1,S,None,primary,True,False,None,Is within-subgroup variation stable?
2,X,None,primary,True,False,None,Is individual variation stable over time?
3,mR,None,primary,True,False,None,Is range variation stable over time?
4,Histogram,None,primary,True,False,None,What shape does the response distribution take?
5,Xbar,R1,residual,True,False,None,How do subgroup means vary about the overall m...
6,X,R1,residual,True,False,None,How do individual values vary about the overal...
7,S,R2,residual,True,False,None,Is within-subgroup variation stable?
8,X,R2,residual,True,False,None,Is within-subgroup variation stable?
9,Xbar,R3,residual,True,False,None,Is there factor×time interaction affecting the...


In [32]:
# Understand why a specific (chart, residual) pair is unavailable
print(study.why_not('S', value='R1'))

'S' with value='R1' is not valid for ADS 1.
Valid charts for R1: Xbar, X


### Design Report

The `study.design()` method returns a `DesignReport` showing the structure of your study -- K (factor groups), T (time points), R (total cells), and N (observations per cell).

In [33]:
# Design report -- observed structure (no plan specified)
report = study.design()
print(report)

Design Report (2 factors)
  Design-state lineage:
    PDS (Planned):    no plan supplied
    SDS (Sampling):   1 (Full Replication)
    ADS (Analytical): 1 (Full Replication)
  Min cell size: 2 | K: 6 | T: 8 | R: 48 | N: (min=2, median=3.0, max=5)

  Factors:
    factor 1: observed=['F1_1', 'F1_2', 'F1_3']
    factor 2: observed=['F2_1', 'F2_2']

  Structure: Complete structure

  Available analyses (ADS 1):
    Primary: Histogram, Xbar *, S, X, mR
    R2: S, X
    R3: Xbar, S
    R4: Xbar, S
    R5: Xbar, S
    R6: Xbar, S
    Methods: Capability, Loss Function, Maximum Information


## 7. Companion Charts

Wheeler recommends reading certain charts as pairs -- the variation chart first (S or R), then the location chart (Xbar or X). The `companion=True` parameter returns both charts in a single result.

**Reading order:**
1. Check the S chart -- is within-group variation stable?
2. Then read the Xbar chart -- only meaningful if S is stable

In [34]:
# Companion Xbar+S -- returns both charts in one result
result_paired = study.execute(chart='Xbar', companion=True)

print(f"Charts in companion result: {result_paired.all_charts}")

# Plot Xbar
result_paired.plot(chart='Xbar', show_stats=True).show()

Charts in companion result: ['Xbar', 'S']


In [35]:
# Plot the companion S chart -- read this first
result_paired.plot(chart='S', show_stats=True).show()

In [36]:
# Companion X+mR stratified by factor 1
result_paired_imr = study.execute(chart='X', by=['factor 1'], companion=True)

print(f"Charts in companion result: {result_paired_imr.all_charts}")

# Plot X
result_paired_imr.plot(chart='X', show_zones=True).show()

Charts in companion result: ['X', 'mR']


In [37]:
# Plot the companion mR chart
result_paired_imr.plot(chart='mR', show_zones=True).show()

## 8. Effects & Interaction Charts

When your study has factors (and optionally time), you can visualize main effects and interactions directly from the analysis result. These charts help answer: *Which factors matter, and do they interact?*

There are five effects chart types:

| Chart | What It Shows |
|-------|---------------|
| `Effects` | All main effects (factor + time) combined |
| `MainEffects` | Factor main effects only |
| `TimeEffects` | Time main effects only |
| `TimeInteraction` | Factor x time interaction |
| `FactorInteraction` | Factor x factor interaction |

In [38]:
# All main effects combined (factor + time)
result.plot(chart='Effects').show()

In [39]:
# Factor main effects only
result.plot(chart='MainEffects').show()

In [40]:
# Time effects only
result.plot(chart='TimeEffects').show()

In [41]:
# Factor x time interaction
result.plot(chart='TimeInteraction').show()

In [42]:
# Factor x factor interaction (requires 2+ factors)
result.plot(chart='FactorInteraction').show()

### Accessing Raw Effects Data

You can also access the underlying effects and interactions data programmatically.

In [43]:
# Raw effects data
print(f"Has effects: {result.has_effects}")
print(f"Has interactions: {result.has_interactions}")

print("\nEffects keys:", list(result.effects.keys()))
print("Interactions keys:", list(result.interactions.keys()))

Has effects: True
Has interactions: True

Effects keys: ['factor 1', 'factor 2', 'main_effect', 'time', 'factor 1_MEs', 'factor 2_MEs', 'factor_interaction_effects']
Interactions keys: ['factor_time', 'factor_factor']


## Summary

### When to Use Each `by` Configuration

| Configuration | Use Case |
|--------------|----------|
| `by=None` (default) | Compare all factor combinations |
| `by=['factor 1']` | Focus on one factor, aggregate the other |
| `by=[]` | Overall process view, collapsed factors |
| `by=['factor 1', 'factor 2']` | Individual charts per combination |

### Key Concepts

1. **Views, Not Recomputation**: The `by` parameter creates views over the same underlying data. Residuals never change.

2. **Lane Boundaries**: When X charts collapse factors, vertical lane boundaries show where factor transitions occur.

3. **Residuals via `value`**: Use `value='R5'` to chart residuals instead of response.

4. **Recentering**: Plain residuals center around zero; `recentered=True` adds the grand mean back (RCR*), putting the chart on the response scale.

5. **Study Inspection**: Use `study.support` to see all available charts, `study.why_not()` to understand constraints, and `study.design()` to inspect the K/T/N/R structure.

6. **Companion Charts**: Use `companion=True` to get both location and variation charts together (Xbar+S or X+mR). Read the variation chart first.

7. **Effects Charts**: Use `result.plot(chart='Effects')` and related chart types to visualize factor and time effects. Requires a study with factors.